In [ ]:
import nltk
nltk.download('gutenberg')
import nltk
from nltk import word_tokenize, pos_tag, ne_chunk
from nltk.chunk import tree2conlltags
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
import re
from nltk.corpus.reader import PlaintextCorpusReader
import csv
import pandas as pd

In [ ]:
import requests

dickensDictionary = {
  "our-mutual-friend": "https://www.gutenberg.org/cache/epub/883/pg883.txt",
  "great-expectations": "https://www.gutenberg.org/cache/epub/1400/pg1400.txt", 
  "bleak-house": "https://www.gutenberg.org/cache/epub/1023/pg1023.txt",
  "great-expectations": "https://www.gutenberg.org/cache/epub/1400/pg1400.txt",
  "hard-times": "https://www.gutenberg.org/cache/epub/786/pg786.txt",
  "christmas-carol": "https://www.gutenberg.org/cache/epub/46/pg46.txt",
  "david-copperfield": "https://www.gutenberg.org/cache/epub/766/pg766.txt",
  "tale-of-two-cities": "https://www.gutenberg.org/cache/epub/98/pg98.txt",
  "oliver-twist": "https://www.gutenberg.org/cache/epub/730/pg730.txt",
  "pickwick-papers": "https://www.gutenberg.org/cache/epub/580/pg580.txt",
  "nicholas-nickleby": "https://www.gutenberg.org/cache/epub/967/pg967.txt",
  "old-curiosity-shop": "https://www.gutenberg.org/cache/epub/700/pg700.txt",
  "martin-chuzzlewit": "https://www.gutenberg.org/cache/epub/968/pg968.txt",
  "dombey-and-son": "https://www.gutenberg.org/cache/epub/821/pg821.txt",
  "barnaby-rudge": "https://www.gutenberg.org/cache/epub/917/pg917.txt",
  "american-notes": "https://www.gutenberg.org/cache/epub/675/pg675.txt",
  "sketches-by-boz": "https://www.gutenberg.org/cache/epub/882/pg882.txt",
  "mudfog-papers": "https://www.gutenberg.org/cache/epub/882/pg882.txt",
  "mystery-of-edwin-drood": "https://www.gutenberg.org/cache/epub/564/pg564.txt",
  "little-dorrit": "https://www.gutenberg.org/cache/epub/963/pg963.txt",
}


In [ ]:
# Open a new file in write-binary mode and write the content of the response to it
for item in dickensDictionary:
  response = requests.get(dickensDictionary[item])
  with open('data/dickens/' + item + '.txt', 'wb') as file:
    file.write(response.content)

In [ ]:
def get_text(filename):
    # Open the file and read the text
    with open('data/dickens/' + filename, 'r') as file:  # Replace 'filename.txt' with your actual filename
        text = file.read()
        
    return text


In [ ]:
def prepare_text(text):
  # tokenize into paragraphs
  paragraphs = text.split("\n\n")
  # remove newlines within paragraphs
  paragraphs = [re.sub(r'[\n]', ' ', doc) for doc in paragraphs]
  return paragraphs
  


In [ ]:
# paragraphs should be a list of paragraphs
def get_tagged_text(paragraphs):
  ## This is meant to be for part of speech tagging
  # Tokenize the text into sentences, then words
  tokens = [word_tokenize(para) for para in paragraphs]
  # Tag the tokens with their part of speech
  pos_tokens = [nltk.pos_tag(tok) for tok in tokens]
  # Chunk the tagged tokens into named entities
  chunked_tokens = [nltk.ne_chunk(tok) for tok in pos_tokens]
  # Convert the trees into IOB tags
  iob_tokens = [tree2conlltags(tok) for tok in chunked_tokens]
  return iob_tokens

In [ ]:
def get_all_names(iob_tokens):
    allNames = []
    for para in iob_tokens:
        names = []
        current_name = []
        for token, pos, chunk in para:
            if chunk == 'B-PERSON':
                if current_name:
                    # If there's a current name, add it to the list of names
                    names.append(' '.join(current_name))
                # Start a new name
                current_name = [token]
            elif chunk == 'I-PERSON':
                # Continue the current name
                current_name.append(token)
            else:
                if current_name:
                    # If there's a current name, add it to the list of names
                    names.append(' '.join(current_name))
                # Reset the current name
                current_name = []

        # If there's a current name left at the end, add it to the list of names
        if current_name:
            names.append(' '.join(current_name))

        allNames.append(names)
    return allNames

In [ ]:
def get_interaction_list(allNames):
    interactionsList = []
    for para in allNames:
        for name in para:
            for name2 in para:
                if name != name2:
                    interactionsList.append((name, name2))
    return interactionsList

In [ ]:
def get_interaction_list_with_paraNum(allNames, bookName):
    interactionsListWithPara = []
    for index, para in enumerate(allNames):
        for name in para:
            for name2 in para:
                if name != name2:
                    interactionsListWithPara.append((name, name2, index, bookName))
    return interactionsListWithPara

In [ ]:
## this function gets interactions of raw strings but records canonical names
## all names here should be a list of tuples of raw named to be matched and then canonical name.
def get_interaction_list_with_paraNum_from_canonical_names(paragraphs, cNameTuples, bookName):
    interactionsListWithPara = []
    ## para is a tuple here with the first item being the raw string and the second item the canoncial name
    for index, para in enumerate(paragraphs):
        usedNames = []
        for name in cNameTuples:
          for name2 in cNameTuples:
            if name2 not in usedNames:
              if name[0] in para and name2[0] in para and name[1] != name2[1]:
                interactionsListWithPara.append((name[1], name2[1], index, bookName))
          #usedNames.append(name)
    return interactionsListWithPara

In [ ]:
def get_unique_interactions(interactionsList):
    return list(set(interactionsList))

In [ ]:
def save_interactions(interactionsList, filename="interactions.csv"):
    # Open the CSV file in write mode
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)

        # Write the header
        writer.writerow(['Item1', 'Item2', 'Time',"Book"])

        # Write the tuples
        for tuple in interactionsList:
            writer.writerow(tuple)

In [ ]:
def save_names(uniqueInteractionsList):
  flat_list = list(set([item for tuple in uniqueInteractionsList for item in tuple]))

In [ ]:
def save_names(flat_list, filename="names.csv"):
  # Open the CSV file in write mode
  with open(filename, 'w', newline='') as f:
      writer = csv.writer(f)

      # Write the header
      writer.writerow(['id'])

      # Write the items
      for item in flat_list:
          writer.writerow([item,"",""])

In [ ]:
def createInteractionCountList(interactionsList):
  # Convert the list of tuples into a DataFrame
  df = pd.DataFrame(interactionsList, columns=['Item1', 'Item2'])

  # Count the number of occurrences of each tuple
  df = df.groupby(['Item1', 'Item2']).size().reset_index(name='Count')

  return df

In [ ]:
def createInteractionCountList2(interactionsList):
  # Convert the list of tuples into a DataFrame
  df = pd.DataFrame(interactionsList, columns=['Item1', 'Item2', "paraNo", "book"])

  # Count the number of occurrences of each tuple
  df = df.groupby(['Item1', 'Item2']).size().reset_index(name='Count')

  return df

In [ ]:
text = get_text("little-dorrit.txt")
paragraphs = prepare_text(text)
cNamesDf = pd.read_csv("data/Dickens/canonicalNames/little-dorrit.csv")
cNameTuples = []
for index, row in cNamesDf.iterrows():
    if row["Canonical Names"] != "?":
      entry = (row["Match Names"], row["Canonical Names"])
      cNameTuples.append(entry)
tokenized_paragraph = [word_tokenize(para) for para in paragraphs]

In [ ]:
interactionsList = get_interaction_list_with_paraNum_from_canonical_names(tokenized_paragraph, cNameTuples,  "little-dorrit")

In [ ]:
interactionsList

In [ ]:
save_interactions(interactionsList, "output/dickensCSV/interactions-canonical-little-dorrit.csv")

In [ ]:
corpus_interactions = []
for key in dickensDictionary.keys():
  text = get_text(key + ".txt")
  paragraphs = prepare_text(text)
  tagged_text = get_tagged_text(paragraphs)
  allNames = get_all_names(tagged_text)
  interactionsList = get_interaction_list_with_paraNum(allNames, key)
  corpus_interactions.append(interactionsList)
  


In [ ]:
corpus_interactions[2][450]

In [ ]:
# save interactions in different files for each book
for item in corpus_interactions:
  save_interactions(item, "output/dickensCSV/interactions-" + item[0][3] + ".csv")

In [ ]:
# save one combined file
combined_interactions = [item for sublist in corpus_interactions for item in sublist]
save_interactions(combined_interactions, "output/dickensCSV/interactions-combined.csv")

In [ ]:
countDf = createInteractionCountList2(interactionsList)

In [ ]:
countDf

In [ ]:
# Assuming df is your DataFrame and it has columns 'Item1', 'Item2' and 'Count'
pivot_table = countDf.pivot_table(index='Item1', columns='Item2', values='Count', aggfunc='sum', fill_value=0)

In [ ]:
pivot_table

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming pivot_table is your frequency matrix
plt.figure(figsize=(10, 8))  # Adjust as needed
sns.heatmap(pivot_table, cmap='Reds')

plt.show()

In [ ]:
# Assuming pivot_table is your pivot table
matrix = pivot_table.values.tolist()

In [ ]:
from sklearn.decomposition import PCA


# Assume embeddings is a 2D numpy array where each row is a word embedding
pca = PCA(n_components=2)  # Reduce to 2 dimensions
embeddings_pca = pca.fit_transform(pivot_table)

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
# Assume words is a list of word strings corresponding to the embeddings
# and embeddings_pca is the 2D PCA-transformed embeddings

plt.figure(figsize=(10, 10))
plt.scatter(embeddings_pca[:, 0], embeddings_pca[:, 1])

# plt.scatter(embeddings_pca[:, 0], embeddings_pca[:, 1])
# plt.yscale('log')
# plt.xscale('log')


for i, word in enumerate(pivot_table.index):
    plt.annotate(word, xy=(embeddings_pca[i, 0], embeddings_pca[i, 1]))
plt.show()